# Comprehensive Ablation Study — Memetic NAS Operator Contribution (3 × 3 Full Suite)

This notebook merges the statistical analysis from `04_ablation_analysis` and the visual comparison from `05_ablation_comparison` into a single authoritative ablation reference.

| ID | Variant | Component removed |
|---|---|---|
| T1 | `full_proposed` | — (Full proposed algorithm) |
| T2 | `no_pso` | Discrete PSO velocity update |
| T3 | `no_ga` | GA crossover / mutation |
| T4 | `no_moead` | MOEA/D decomposition framework |
| T5 | `no_sa` | Simulated Annealing local refinement |
| T6 | `no_restart` | Population diversity restart |

**Coverage:** 3 datasets × 3 hardware = 9 (dataset × hardware) combinations  
**Statistics:** Wilcoxon rank-sum (Bonferroni-corrected, α = 0.05, 5 comparisons per combo)  
**Visuals:** Seaborn HV/IGD grids · Significance heatmap · Component contribution · 3×3 Pareto overlay

In [1]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
for _parent in [_cwd, *_cwd.parents]:
    if (_parent / "src").exists() and (_parent / "requirements.txt").exists():
        PROJECT_ROOT = _parent
        break
else:
    PROJECT_ROOT = _cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import ranksums

from src.analysis.pareto_metrics import (
    archive_to_points, get_pareto_front, calc_hv, calc_igd, proxy_pareto,
)
from src.analysis.data_loader import (
    DATASETS, HARDWARE,
    load_ablation_archives as _load_ablation_archives,
)
from src.utils.pareto_math import compute_bounds, normalise_to_bounds

ABLATION_DIR = PROJECT_ROOT / "results" / "ablations"
OUT_DIR      = PROJECT_ROOT / "results" / "figures" / "ablation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi":        150,
    "font.size":         10,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

# ── Experiment catalogue ─────────────────────────────────────────────────────
EXPERIMENT_ORDER = [
    ("full_proposed", "T1: Full"),
    ("no_pso",        "T2: No PSO"),
    ("no_ga",         "T3: No GA"),
    ("no_moead",      "T4: No MOEA/D"),
    ("no_sa",         "T5: No SA"),
    ("no_restart",    "T6: No Restart"),
]
EXP_NAMES  = [n for n, _ in EXPERIMENT_ORDER]
EXP_LABELS = {n: l for n, l in EXPERIMENT_ORDER}
EXP_SHORT  = {n: l.split(": ")[1] for n, l in EXPERIMENT_ORDER}

COLORS = {
    "full_proposed": "#1f77b4",
    "no_pso":        "#ff7f0e",
    "no_ga":         "#2ca02c",
    "no_moead":      "#d62728",
    "no_sa":         "#9467bd",
    "no_restart":    "#8c564b",
}
PALETTE = {EXP_SHORT[n]: COLORS[n] for n in EXP_NAMES}

DS_LABEL = {
    "cifar10":        "CIFAR-10",
    "cifar100":       "CIFAR-100",
    "ImageNet16-120": "ImageNet-16",
}
HW_LABEL = {
    "edgegpu_latency": "EdgeGPU",
    "raspi4_latency":  "RPi-4",
    "eyeriss_latency": "Eyeriss",
}

REF = np.array([1.1, 1.1])
print("Setup complete.")

PROJECT_ROOT = /home/abdeslem/Folders/2CS/S2/PROJ/HW-NAS-Memetic
Setup complete.


In [ ]:
# ── Cell 2: Load archives + compute per-seed HV/IGD → tidy DataFrame ─────────
available    = {ds: {hw: {} for hw in HARDWARE} for ds in DATASETS}
all_metrics  = {ds: {hw: {} for hw in HARDWARE} for ds in DATASETS}
tidy_records = []

_total_combos = len(DATASETS) * len(HARDWARE)
_combo_idx    = 0

print("Loading ablation archives …\n")
for ds in DATASETS:
    for hw in HARDWARE:
        _combo_idx += 1
        combo_label = f"{DS_LABEL[ds]:<12} × {HW_LABEL[hw]:<8}"
        print(f"  [{_combo_idx}/{_total_combos}] {combo_label} …", end=" ", flush=True)

        combo_dir = ABLATION_DIR / ds / hw
        for name in EXP_NAMES:
            available[ds][hw][name] = (
                _load_ablation_archives(combo_dir, name)
                if combo_dir.exists() else []
            )

        all_runs = [a for n in EXP_NAMES for a in available[ds][hw][n]]
        if not all_runs:
            print("SKIP (no data)")
            continue

        try:
            p_star = proxy_pareto(all_runs)
        except Exception as exc:
            print(f"SKIP (proxy_pareto: {exc})")
            continue

        bmin, brange = compute_bounds(p_star)
        p_star_n     = normalise_to_bounds(p_star, bmin, brange)

        for name in EXP_NAMES:
            arcs = available[ds][hw][name]
            hvs, igds = [], []
            for seed_i, run in enumerate(arcs):
                pts   = archive_to_points(run)
                front = get_pareto_front(pts)
                nf    = normalise_to_bounds(front, bmin, brange)
                hvs.append(calc_hv(nf, ref_point=REF))
                igds.append(calc_igd(nf, p_star_n))
                tidy_records.append({
                    "Dataset":  DS_LABEL[ds],
                    "Hardware": HW_LABEL[hw],
                    "ds_key":   ds,
                    "hw_key":   hw,
                    "exp_name": name,
                    "Label":    EXP_LABELS[name],
                    "Short":    EXP_SHORT[name],
                    "Seed":     seed_i,
                    "HV":       hvs[-1],
                    "IGD":      igds[-1],
                })
            all_metrics[ds][hw][name] = {
                "hv":  np.array(hvs),
                "igd": np.array(igds),
            }
        n_seeds = len(available[ds][hw]["full_proposed"])
        print(f"✓  ({n_seeds} seeds/exp)")

tidy_df = pd.DataFrame(tidy_records)
SHORT_ORDER = [EXP_SHORT[n] for n in EXP_NAMES]
tidy_df["Short"] = pd.Categorical(tidy_df["Short"], categories=SHORT_ORDER, ordered=True)
print(f"\nTidy DataFrame: {len(tidy_df)} rows")

Loading ablation archives …

  [1/9] CIFAR-10     × EdgeGPU  … 
Compiled modules for significant speedup can not be used!
https://pymoo.org/installation.html#installation

To disable this warning:
from pymoo.config import Config
Config.warnings['not_compiled'] = False

✓  (0 seeds/exp)
  [2/9] CIFAR-10     × RPi-4    … ✓  (0 seeds/exp)
  [3/9] CIFAR-10     × Eyeriss  … ✓  (0 seeds/exp)
  [4/9] CIFAR-100    × EdgeGPU  … ✓  (0 seeds/exp)
  [5/9] CIFAR-100    × RPi-4    … ✓  (0 seeds/exp)
  [6/9] CIFAR-100    × Eyeriss  … ✓  (0 seeds/exp)
  [7/9] ImageNet-16  × EdgeGPU  … ✓  (0 seeds/exp)
  [8/9] ImageNet-16  × RPi-4    … ✓  (0 seeds/exp)
  [9/9] ImageNet-16  × Eyeriss  … 

In [ ]:
# ── Cell 3: Summary pivot table (mean HV/IGD across 9 combos) ────────────────
records = []
for ds in DATASETS:
    for hw in HARDWARE:
        for name in EXP_NAMES:
            m    = all_metrics[ds][hw].get(name, {})
            hvs  = m.get("hv",  np.array([]))
            igds = m.get("igd", np.array([]))
            records.append({
                "Dataset":    DS_LABEL[ds],
                "Hardware":   HW_LABEL[hw],
                "Experiment": EXP_LABELS[name],
                "exp_name":   name,
                "hv_mean":    hvs.mean()  if len(hvs)  else float("nan"),
                "hv_std":     hvs.std()   if len(hvs)  else float("nan"),
                "igd_mean":   igds.mean() if len(igds) else float("nan"),
                "igd_std":    igds.std()  if len(igds) else float("nan"),
                "n_seeds":    len(hvs),
            })

df_pivot = pd.DataFrame(records)

pivot_hv = df_pivot.pivot_table(
    index="Experiment",
    columns=["Dataset", "Hardware"],
    values="hv_mean",
    aggfunc="mean",
).reindex([EXP_LABELS[n] for n in EXP_NAMES])

pivot_igd = df_pivot.pivot_table(
    index="Experiment",
    columns=["Dataset", "Hardware"],
    values="igd_mean",
    aggfunc="mean",
).reindex([EXP_LABELS[n] for n in EXP_NAMES])

print("Mean Hypervolume ↑ (9 combinations):\n")
display(pivot_hv.round(4).style
        .highlight_max(axis=0, color="#cce5ff")
        .set_caption("HV ↑ — higher is better"))

print("\nMean IGD ↓ (9 combinations):\n")
display(pivot_igd.round(4).style
        .highlight_min(axis=0, color="#d4edda")
        .set_caption("IGD ↓ — lower is better"))

In [ ]:
# ── Cell 4: Wilcoxon rank-sum tests — T1 vs T2–T6 (Bonferroni-corrected) ─────
N_TESTS = len(EXP_NAMES) - 1  # 5 comparisons per combo

stat_records = []
for ds in DATASETS:
    for hw in HARDWARE:
        hv1 = all_metrics[ds][hw].get("full_proposed", {}).get("hv", np.array([]))
        if len(hv1) == 0:
            continue
        for name in EXP_NAMES[1:]:
            hv2 = all_metrics[ds][hw].get(name, {}).get("hv", np.array([]))
            if len(hv2) == 0:
                p_raw, stat = 1.0, 0.0
            else:
                try:
                    stat, p_raw = ranksums(hv1, hv2)
                except ValueError:
                    p_raw, stat = 1.0, 0.0
            p_bonf = min(p_raw * N_TESTS, 1.0)
            sig    = ("***" if p_bonf < 0.001 else
                      "**"  if p_bonf < 0.01  else
                      "*"   if p_bonf < 0.05  else "n.s.")
            stat_records.append({
                "Dataset":  DS_LABEL[ds],
                "Hardware": HW_LABEL[hw],
                "Ablation": EXP_LABELS[name],
                "exp_name": name,
                "ds_key":   ds,
                "hw_key":   hw,
                "W":        round(float(stat), 3),
                "p_raw":    float(p_raw),
                "p_bonf":   float(p_bonf),
                "sig":      sig,
                "delta_hv": float(hv1.mean() - hv2.mean()) if len(hv2) else float("nan"),
            })

df_stats = pd.DataFrame(stat_records)

pivot_sig = df_stats.pivot_table(
    index="Ablation",
    columns=["Dataset", "Hardware"],
    values="sig",
    aggfunc="first",
).reindex([EXP_LABELS[n] for n in EXP_NAMES[1:]])

print("Wilcoxon rank-sum (T1 Full vs each ablation)  Bonferroni-corrected:\n")
print(pivot_sig.to_string())
print("\n*** p<0.001   ** p<0.01   * p<0.05   n.s. = not significant")

In [ ]:
# ── Cell 5: 3×3 Hypervolume seaborn boxplot + stripplot grid ─────────────────
fig_hv, axes_hv = plt.subplots(3, 3, figsize=(20, 14), constrained_layout=True)
fig_hv.suptitle(
    "Ablation Study — Hypervolume ↑  (per-seed distribution)\n"
    "3 datasets × 3 hardware  |  Dashed line = T1 Full median",
    fontsize=13, fontweight="bold",
)

for r, ds in enumerate(DATASETS):
    for c, hw in enumerate(HARDWARE):
        ax  = axes_hv[r, c]
        sub = tidy_df[(tidy_df["ds_key"] == ds) & (tidy_df["hw_key"] == hw)]
        order = [s for s in SHORT_ORDER if s in sub["Short"].cat.categories
                 and len(sub[sub["Short"] == s]) > 0]

        if sub.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center",
                    transform=ax.transAxes, fontsize=10, color="gray")
            ax.set_title(f"{DS_LABEL[ds]}  ×  {HW_LABEL[hw]}", fontsize=9)
            continue

        sns.boxplot(
            data=sub, x="Short", y="HV", order=order,
            palette=PALETTE, ax=ax,
            showfliers=False, linewidth=0.9,
            boxprops=dict(alpha=0.75),
        )
        sns.stripplot(
            data=sub, x="Short", y="HV", order=order,
            color="black", size=2.2, alpha=0.35, jitter=True, ax=ax,
        )

        t1_med = sub[sub["exp_name"] == "full_proposed"]["HV"].median()
        ax.axhline(t1_med, color=COLORS["full_proposed"], lw=1.0,
                   ls="--", alpha=0.55, zorder=0)

        ax.set_title(f"{DS_LABEL[ds]}  ×  {HW_LABEL[hw]}", fontsize=9)
        ax.set_xlabel("")
        ax.set_xticklabels(order, fontsize=7.5, rotation=30, ha="right")
        ax.set_ylabel("Hypervolume ↑" if c == 0 else "", fontsize=8)
        ax.tick_params(axis="y", labelsize=7.5)

save_hv = OUT_DIR / "ablation_hv_3x3.pdf"
plt.savefig(save_hv, bbox_inches="tight")
plt.show()
print(f"Saved: {save_hv}")

In [ ]:
# ── Cell 6: 3×3 IGD seaborn boxplot + stripplot grid ─────────────────────────
fig_igd, axes_igd = plt.subplots(3, 3, figsize=(20, 14), constrained_layout=True)
fig_igd.suptitle(
    "Ablation Study — IGD ↓  (per-seed distribution)\n"
    "3 datasets × 3 hardware  |  Dashed line = T1 Full median",
    fontsize=13, fontweight="bold",
)

for r, ds in enumerate(DATASETS):
    for c, hw in enumerate(HARDWARE):
        ax  = axes_igd[r, c]
        sub = tidy_df[(tidy_df["ds_key"] == ds) & (tidy_df["hw_key"] == hw)]
        order = [s for s in SHORT_ORDER if s in sub["Short"].cat.categories
                 and len(sub[sub["Short"] == s]) > 0]

        if sub.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center",
                    transform=ax.transAxes, fontsize=10, color="gray")
            ax.set_title(f"{DS_LABEL[ds]}  ×  {HW_LABEL[hw]}", fontsize=9)
            continue

        sns.boxplot(
            data=sub, x="Short", y="IGD", order=order,
            palette=PALETTE, ax=ax,
            showfliers=False, linewidth=0.9,
            boxprops=dict(alpha=0.75),
        )
        sns.stripplot(
            data=sub, x="Short", y="IGD", order=order,
            color="black", size=2.2, alpha=0.35, jitter=True, ax=ax,
        )

        t1_med = sub[sub["exp_name"] == "full_proposed"]["IGD"].median()
        ax.axhline(t1_med, color=COLORS["full_proposed"], lw=1.0,
                   ls="--", alpha=0.55, zorder=0)

        ax.set_title(f"{DS_LABEL[ds]}  ×  {HW_LABEL[hw]}", fontsize=9)
        ax.set_xlabel("")
        ax.set_xticklabels(order, fontsize=7.5, rotation=30, ha="right")
        ax.set_ylabel("IGD ↓" if c == 0 else "", fontsize=8)
        ax.tick_params(axis="y", labelsize=7.5)

save_igd = OUT_DIR / "ablation_igd_3x3.pdf"
plt.savefig(save_igd, bbox_inches="tight")
plt.show()
print(f"Saved: {save_igd}")

In [ ]:
# ── Cell 7: Component contribution ranking (HV drop + IGD rise vs T1) ─────────
contrib_records = []
for name in EXP_NAMES[1:]:
    for ds in DATASETS:
        for hw in HARDWARE:
            hv1  = all_metrics[ds][hw].get("full_proposed", {}).get("hv",  np.array([]))
            hv2  = all_metrics[ds][hw].get(name, {}).get("hv",  np.array([]))
            igd1 = all_metrics[ds][hw].get("full_proposed", {}).get("igd", np.array([]))
            igd2 = all_metrics[ds][hw].get(name, {}).get("igd", np.array([]))
            if len(hv1) > 0 and len(hv2) > 0:
                contrib_records.append({
                    "Ablation":  EXP_LABELS[name],
                    "Short":     EXP_SHORT[name],
                    "exp_name":  name,
                    "Dataset":   DS_LABEL[ds],
                    "Hardware":  HW_LABEL[hw],
                    "delta_hv":  float(hv1.mean() - hv2.mean()),
                    "delta_igd": float(
                        igd2.mean() - igd1.mean() if len(igd1) > 0 and len(igd2) > 0 else float("nan")
                    ),
                })

contrib_df  = pd.DataFrame(contrib_records)
mean_contrib = (
    contrib_df.groupby(["exp_name", "Short", "Ablation"])[["delta_hv", "delta_igd"]]
    .mean()
    .reset_index()
    .sort_values("delta_hv", ascending=False)
)

fig_c, axes_c = plt.subplots(1, 2, figsize=(14, 4.8))

for ax, (metric, xlabel, title_sfx) in zip(axes_c, [
    ("delta_hv",  "Mean HV drop  (T1 − Ti)  ↑ = more critical",  "HV perspective"),
    ("delta_igd", "Mean IGD rise  (Ti − T1)  ↑ = more critical",  "IGD perspective"),
]):
    bars = ax.barh(
        mean_contrib["Short"],
        mean_contrib[metric],
        color=[COLORS[n] for n in mean_contrib["exp_name"]],
        edgecolor="white", linewidth=0.8, alpha=0.85,
    )
    ax.axvline(0, color="black", lw=0.8)
    ref = mean_contrib[metric].abs().max()
    for bar, val in zip(bars, mean_contrib[metric]):
        ax.text(val + ref * 0.015, bar.get_y() + bar.get_height() / 2,
                f"{val:+.4f}", va="center", fontsize=8.5)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_title(f"Component Contribution\n({title_sfx})", fontweight="bold", fontsize=10)
    ax.invert_yaxis()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig_c.tight_layout()
save_c = OUT_DIR / "ablation_component_contribution.pdf"
plt.savefig(save_c, bbox_inches="tight")
plt.show()
print(f"Saved: {save_c}")
print("\nRanking (HV):")
print(mean_contrib[["Ablation", "delta_hv", "delta_igd"]].to_string(index=False))

In [ ]:
# ── Cell 8: Significance heatmap (ablation × dataset/hardware combo) ──────────
ablation_lbls = [EXP_LABELS[n] for n in EXP_NAMES[1:]]
combo_lbls    = [f"{DS_LABEL[ds]}\n{HW_LABEL[hw]}"
                 for ds in DATASETS for hw in HARDWARE]

p_matrix     = np.ones((len(ablation_lbls), len(combo_lbls)))
delta_matrix = np.zeros_like(p_matrix)
for j, (ds, hw) in enumerate([(d, h) for d in DATASETS for h in HARDWARE]):
    for i, name in enumerate(EXP_NAMES[1:]):
        row = df_stats[(df_stats["ds_key"] == ds) &
                       (df_stats["hw_key"] == hw) &
                       (df_stats["exp_name"] == name)]
        if not row.empty:
            p_matrix[i, j]     = row.iloc[0]["p_bonf"]
            delta_matrix[i, j] = row.iloc[0]["delta_hv"]

def _sig_color(p):
    if   p < 0.001: return "#1a7dc8"
    elif p < 0.01:  return "#5baee0"
    elif p < 0.05:  return "#aad4f0"
    else:           return "#e8e8e8"

fig_h, ax_h = plt.subplots(figsize=(14, 4.5))
nr, nc = p_matrix.shape
for i in range(nr):
    for j in range(nc):
        color = _sig_color(p_matrix[i, j])
        rect  = plt.Rectangle([j - 0.5, i - 0.5], 1, 1,
                               facecolor=color, edgecolor="white", lw=1.5)
        ax_h.add_patch(rect)
        if p_matrix[i, j] < 0.05:
            stars = ("***" if p_matrix[i, j] < 0.001 else
                     "**"  if p_matrix[i, j] < 0.01  else "*")
            ax_h.text(j, i + 0.18, stars,
                      ha="center", va="center",
                      fontsize=9, fontweight="bold", color="white")
        ax_h.text(j, i - 0.22, f"Δ{delta_matrix[i, j]:+.3f}",
                  ha="center", va="center", fontsize=6.5, color="#333")

ax_h.set_xlim(-0.5, nc - 0.5)
ax_h.set_ylim(-0.5, nr - 0.5)
ax_h.set_xticks(range(nc))
ax_h.set_xticklabels(combo_lbls, fontsize=7.5)
ax_h.set_yticks(range(nr))
ax_h.set_yticklabels(ablation_lbls, fontsize=9)
ax_h.set_title(
    "T1 Full Proposed vs Ablation Variants — Wilcoxon Significance Heatmap\n"
    "Colour = Bonferroni-corrected p-value  |  Δ = mean HV gain of T1",
    fontsize=10,
)
legend_patches = [
    mpatches.Patch(facecolor="#1a7dc8", label="p < 0.001  (***)"),
    mpatches.Patch(facecolor="#5baee0", label="p < 0.01   (**)"),
    mpatches.Patch(facecolor="#aad4f0", label="p < 0.05   (*)"),
    mpatches.Patch(facecolor="#e8e8e8", label="n.s."),
]
ax_h.legend(handles=legend_patches, loc="upper right",
            bbox_to_anchor=(1.22, 1.0), fontsize=8)
fig_h.tight_layout()
save_hm = OUT_DIR / "ablation_significance_heatmap.pdf"
plt.savefig(save_hm, bbox_inches="tight")
plt.show()
print(f"Saved: {save_hm}")

In [ ]:
# ── Cell 9: 3×3 Aggregate Pareto front overlay (T1 drawn on top) ──────────────
fig_p, axes_p = plt.subplots(3, 3, figsize=(20, 14), constrained_layout=True)
fig_p.suptitle(
    "Ablation Study — Aggregate Pareto Front Overlay (all seeds pooled)\n"
    "3 datasets × 3 hardware  |  T1 Full Proposed drawn on top",
    fontsize=13, fontweight="bold",
)

legend_built = False

for r, ds in enumerate(DATASETS):
    for c, hw in enumerate(HARDWARE):
        ax = axes_p[r, c]

        all_acc, all_lat, lines = [], [], []
        # Draw ablations first, then T1 on top
        for name in EXP_NAMES[1:] + ["full_proposed"]:
            arcs = available[ds][hw].get(name, [])
            if not arcs:
                continue
            try:
                pts_all = np.vstack([archive_to_points(a) for a in arcs])
            except ValueError:
                continue
            front = get_pareto_front(pts_all)
            acc   = -front[:, 0]
            lat   =  front[:, 1]
            order = np.argsort(acc)
            acc, lat = acc[order], lat[order]
            all_acc.append(acc)
            all_lat.append(lat)
            lines.append((name, acc, lat))

        if not lines:
            ax.text(0.5, 0.5, "No data", ha="center", va="center",
                    transform=ax.transAxes, fontsize=10, color="gray")
            ax.set_title(f"{DS_LABEL[ds]}  ×  {HW_LABEL[hw]}", fontsize=8)
            continue

        x_all = np.concatenate(all_acc)
        y_all = np.concatenate(all_lat)
        xrng  = max(x_all.max() - x_all.min(), 0.1)
        yrng  = max(y_all.max() - y_all.min(), 1e-6)
        ax.set_xlim(x_all.min() - xrng * 0.03, x_all.max() + xrng * 0.03)
        ax.set_ylim(y_all.min() - yrng * 0.05, y_all.max() + yrng * 0.08)

        for name, acc, lat in lines:
            is_t1 = (name == "full_proposed")
            ax.step(acc, lat, where="post",
                    color=COLORS[name],
                    linewidth=2.4 if is_t1 else 1.1,
                    alpha=1.0 if is_t1 else 0.65,
                    label=EXP_SHORT[name],
                    zorder=10 if is_t1 else 4)
            ax.scatter(acc, lat,
                       s=18 if is_t1 else 10,
                       color=COLORS[name],
                       alpha=1.0 if is_t1 else 0.65,
                       zorder=11 if is_t1 else 5)

        ax.set_title(f"{DS_LABEL[ds]}  ×  {HW_LABEL[hw]}", fontsize=9)
        if c == 0:
            ax.set_ylabel("Latency", fontsize=8)
        if r == 2:
            ax.set_xlabel("Accuracy (%)", fontsize=8)

        if not legend_built:
            ax.legend(fontsize=7, loc="upper right", ncol=2,
                      framealpha=0.88, handlelength=1.4)
            legend_built = True

save_p = OUT_DIR / "ablation_pareto_3x3.pdf"
plt.savefig(save_p, bbox_inches="tight")
plt.show()
print(f"Saved: {save_p}")